In [ ]:
# -----------------------------
# Imports
# -----------------------------

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [5]:
df = pd.read_csv('train.csv')

In [6]:
# -----------------------------
# Custom Feature Transformer
# -----------------------------

class CustomFeatures(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        self.monthly_charge_median_ = X['MonthlyCharges'].median()
        return self

    def transform(self, X):
        X = X.copy()

        X['NewCustomer'] = (X['tenure'] <= 3).astype(int)
        X['LoyalCustomer'] = (X['tenure'] >= 50).astype(int)

        service_cols = [
            'PhoneService',
            'OnlineSecurity',
            'OnlineBackup',
            'DeviceProtection',
            'TechSupport',
            'StreamingTV',
            'StreamingMovies'
        ]

        X['TotalServices'] = (X[service_cols] == 'Yes').sum(axis=1)

        X['HighMonthlyCharges'] = (
            X['MonthlyCharges'] > self.monthly_charge_median_
        ).astype(int)

        X['AvgChargesPerMonth'] = (
            X['TotalCharges'] / (X['tenure'] + 1)
        )

        X['AutoPayment'] = (
            X['PaymentMethod']
            .str.contains('automatic', case=False)
            .astype(int)
        )

        X['FiberUser'] = (
            X['InternetService'] == 'Fiber optic'
        ).astype(int)

        X['SecurityServices'] = (
            X[
                ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
            ] == 'Yes'
        ).sum(axis=1)

        return X

In [7]:
# -----------------------------
# Train-Test Split
# -----------------------------

X = df.drop(columns=['Churn'])
y = df['Churn'].map({
    'No': 0,
    'Yes': 1
})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
# -----------------------------
# Columns After Feature Engineering
# -----------------------------

num_cols = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'TotalServices',
    'AvgChargesPerMonth',
    'SecurityServices'
]

binary_cols = [
    'SeniorCitizen',
    'NewCustomer',
    'LoyalCustomer',
    'HighMonthlyCharges',
    'AutoPayment',
    'FiberUser'
]

cat_cols = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod'
]

In [9]:
# -----------------------------
# Preprocessing
# -----------------------------

numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

category_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessing = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', category_pipeline, cat_cols),
    ('binary', 'passthrough', binary_cols)
])

In [10]:
# -----------------------------
# Tuned XGBoost Pipeline
# -----------------------------

xgb_pipe = Pipeline([
    ('CustomFeatures', CustomFeatures()),
    ('preprocessing', preprocessing),
    ('model', XGBClassifier(
        n_estimators=560,
        learning_rate=0.21609223800887833,
        max_depth=3,
        subsample=0.7901480892728447,
        colsample_bytree=0.7024273291045295,
        gamma=0.20216794769215674,
        min_child_weight=2,
        reg_alpha=0.4034384046707924,
        reg_lambda=2.687290787020558,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [11]:
# -----------------------------
# Fit + Evaluate
# -----------------------------

xgb_pipe.fit(X_train, y_train)

y_prob = xgb_pipe.predict_proba(X_test)[:, 1]

threshold = 0.35
y_pred = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8484756687619385
Precision: 0.6304606949162643
Recall: 0.7905317042185106
F1 Score: 0.7014804131231246
ROC-AUC: 0.9169234779233602

Confusion Matrix:
[[79675 12401]
 [ 5606 21157]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.87      0.90     92076
           1       0.63      0.79      0.70     26763

    accuracy                           0.85    118839
   macro avg       0.78      0.83      0.80    118839
weighted avg       0.87      0.85      0.85    118839



In [12]:
# -----------------------------
# Threshold Comparison
# -----------------------------

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

for threshold in thresholds:
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Threshold: 0.3
Precision: 0.60375768122247
Recall: 0.8296902439935732
F1: 0.6989188083284809
ROC-AUC: 0.9169234779233602

Threshold: 0.35
Precision: 0.6304606949162643
Recall: 0.7905317042185106
F1: 0.7014804131231246
ROC-AUC: 0.9169234779233602

Threshold: 0.4
Precision: 0.6590639810426541
Recall: 0.748234502858424
F1: 0.7008241902462071
ROC-AUC: 0.9169234779233602

Threshold: 0.45
Precision: 0.6853620166099587
Recall: 0.6999588984792438
F1: 0.692583555161195
ROC-AUC: 0.9169234779233602

Threshold: 0.5
Precision: 0.7145471941428512
Recall: 0.6418189291185592
F1: 0.6762332191645998
ROC-AUC: 0.9169234779233602


In [14]:
# -----------------------------
# Final Kaggle Submission Block
# -----------------------------

train_final = pd.read_csv("train.csv")
test_final = pd.read_csv("test.csv")

test_ids = test_final["id"] if "id" in test_final.columns else test_final.iloc[:, 0]

train_final = train_final.drop(columns=["id"], errors="ignore")
test_final = test_final.drop(columns=["id"], errors="ignore")

X_final = train_final.drop(columns=["Churn"])
y_final = train_final["Churn"].map({
    "No": 0,
    "Yes": 1
})

xgb_pipe.fit(X_final, y_final)

test_prob = xgb_pipe.predict_proba(test_final)[:, 1]

threshold = 0.35
test_pred = (test_prob >= threshold).astype(int)

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_pred
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully")
print(submission["Churn"].value_counts())
print(submission.head())

submission.csv created successfully
Churn
0    185237
1     69418
Name: count, dtype: int64
       id  Churn
0  594194      0
1  594195      0
2  594196      0
3  594197      0
4  594198      1
